Exploratory notebook: outputs and row-level examples are intentionally omitted from Git. This notebook predates the audited Purchase-only 12-month label; use docs/data-contract.md and scripts/label/label_rules.py for the current study. Run only with local Freddie Mac data; it is not a synthetic quick-start.

# EDA bảng `orig` (thông tin khởi tạo khoản vay), DuckDB

Notebook này **chỉ làm việc với `orig`** (~12,8 triệu dòng, 1 dòng/khoản vay), chạy toàn bộ được trên máy thường.
Phần `perf` và tỷ lệ vỡ nợ nằm ở notebook riêng: `eda_perf.ipynb`.

**Quy trình:** 1. Cấu trúc → 2. Chất lượng → 3. Đơn biến → 4. Hai biến / nhiều biến → 5. Insight

**Cách chạy:** chạy từ trên xuống. Chạy tới ô 1.6 thì kiểm tra tên cột (không được có dòng `⚠️`). Muốn thử nhanh, đặt `ORIG_QUARTER = "2023Q1"`.

## 0. Setup

In [ ]:
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

# ====== CẤU HÌNH ======
from pathlib import Path
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/pilot_baseline.yaml").is_file())
DATA_DIR = str(PROJECT_ROOT / "parquet")
MEMORY_LIMIT = "6GB"
THREADS      = 4
ORIG_QUARTER = None      # None = toàn bộ; hoặc "2023Q1" để thử nhanh

In [ ]:
con = duckdb.connect("eda_orig.duckdb")
con.execute(f"SET memory_limit='{MEMORY_LIMIT}'; SET threads={THREADS}; SET temp_directory='./duckdb_tmp';")

def q(sql):
    return con.execute(sql).df()

con.execute(f"""
    CREATE OR REPLACE VIEW orig AS
    SELECT * FROM read_parquet('{DATA_DIR}/orig/*/data.parquet', hive_partitioning=true, union_by_name=true)
""")

# Mã "không có dữ liệu" của Freddie Mac -> NULL (đối chiếu user guide)
SPECIAL_CODES = {"credit_score": 9999, "vantage_score_4": 9999,
                 "dti": 999, "ltv": 999, "cltv": 999, "mi_percentage": 999}
_cols = set(q("DESCRIBE orig")["column_name"])
SPECIAL_CODES = {c: v for c, v in SPECIAL_CODES.items() if c in _cols}
repl = ", ".join(f'NULLIF("{c}", {v}) AS "{c}"' for c, v in SPECIAL_CODES.items())
w = f"WHERE quarter = '{ORIG_QUARTER}'" if ORIG_QUARTER else ""
con.execute(f"CREATE OR REPLACE VIEW orig_s AS SELECT * REPLACE ({repl}) FROM orig {w}")

print("Phạm vi:", ORIG_QUARTER or "TOÀN BỘ orig")
print("Số dòng chứa mã đặc biệt (đã đổi thành NULL trong orig_s):")
display(q("SELECT " + ", ".join(f'COUNT(*) FILTER (WHERE "{c}" = {v}) AS "{c}"' for c, v in SPECIAL_CODES.items()) + f" FROM orig {w}"))

## Bước 1. Hiểu cấu trúc dữ liệu

### 1.1 Tên cột và kiểu dữ liệu

In [ ]:
desc_orig = q("DESCRIBE orig")
display(desc_orig[["column_name", "column_type", "null"]])

### 1.2 Số khoản vay theo quý

In [ ]:
rows = q("SELECT quarter, COUNT(*) AS n_orig FROM orig GROUP BY 1 ORDER BY 1")
display(rows); print("Tổng:", f"{rows.n_orig.sum():,.0f}")
rows.plot.bar(x="quarter", y="n_orig", legend=False, title="Số khoản vay (orig) theo quý"); plt.show()

### 1.3 Xem mẫu

In [ ]:
display(q("SELECT * FROM orig_s LIMIT 5"))

### 1.4 Schema có đồng nhất giữa các quý không
*(cột có > 1 kiểu, hoặc thiếu ở vài quý, cần xử lý)*

In [ ]:
r = q(f"""
    SELECT name, COUNT(DISTINCT type) AS so_kieu, list(DISTINCT type) AS cac_kieu, COUNT(DISTINCT file_name) AS so_file
    FROM parquet_schema('{DATA_DIR}/orig/*/data.parquet') WHERE name <> 'schema'
    GROUP BY name ORDER BY so_kieu DESC, name""")
display(r[(r.so_kieu > 1) | (r.so_file < r.so_file.max())])
print("Số cột lệch:", int(((r.so_kieu > 1) | (r.so_file < r.so_file.max())).sum()))

### 1.5 SUMMARIZE

In [ ]:
display(q("SUMMARIZE orig_s"))

### 1.6 CẤU HÌNH CỘT (sửa nếu tên cột khác)

In [ ]:
LOAN_ID       = "loan_id"
BREAKDOWN_COL = "credit_score"

# Cột là MÃ hoặc NGÀY (dù lưu dạng số) -> không đưa vào thống kê / histogram / tương quan
CODE_COLS = ["first_payment_date", "maturity_date", "msa", "vantage_score_4",
             "original_loan_term", "pre_harp_loan_id"]

NUM_TYPES = ("TINYINT","SMALLINT","INTEGER","BIGINT","HUGEINT","UTINYINT","USMALLINT",
             "UINTEGER","UBIGINT","FLOAT","DOUBLE","DECIMAL")
def cols_of(desc, kind):
    if kind == "num":
        m = desc.column_type.str.upper().str.startswith(NUM_TYPES)
    else:
        m = desc.column_type.str.upper().isin(["VARCHAR", "BOOLEAN"])
    return desc.loc[m, "column_name"].tolist()

orig_num = [c for c in cols_of(desc_orig, "num") if c not in CODE_COLS]
orig_cat = [c for c in cols_of(desc_orig, "cat") if c not in ("quarter", LOAN_ID, "pre_harp_loan_id")]

def has(col): return col in desc_orig.column_name.values
for col in [LOAN_ID, BREAKDOWN_COL]:
    if not has(col): print(f"⚠️ Không thấy cột '{col}' trong orig, hãy sửa cấu hình ở trên")
print("Cột số:", orig_num); print("Cột phân loại:", orig_cat)

**Data dictionary** (ghi ý nghĩa từng cột):

| Cột | Kiểu | Ý nghĩa | Ghi chú |
|---|---|---|---|
| | | | |

## Bước 2. Kiểm tra chất lượng dữ liệu

### 2.1 Giá trị thiếu (NULL)

In [ ]:
cols = desc_orig["column_name"].tolist()
sel = ", ".join(f'COUNT(*) - COUNT("{c}") AS "{c}"' for c in cols)
r = q(f"SELECT COUNT(*) AS __total, {sel} FROM orig_s")
total = int(r.pop("__total").iloc[0])
nr = r.T.reset_index(); nr.columns = ["cot", "so_null"]; nr["pct_null"] = (nr.so_null / total * 100).round(2)
display(nr[nr.so_null > 0].sort_values("pct_null", ascending=False).reset_index(drop=True))

### 2.2 Mã đặc biệt: min / max và giá trị phổ biến

In [ ]:
def top_values(col, k=5):
    return q(f"""SELECT "{col}" AS gia_tri, COUNT(*) AS n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 2) AS pct
                 FROM orig_s GROUP BY 1 ORDER BY n DESC LIMIT {k}""")

mm = q("SELECT " + ", ".join(f'MIN("{c}") AS "{c}__min", MAX("{c}") AS "{c}__max"' for c in orig_num) + " FROM orig_s")
display(pd.DataFrame({"min": [mm[f"{c}__min"][0] for c in orig_num], "max": [mm[f"{c}__max"][0] for c in orig_num]}, index=orig_num))
for c in orig_cat:
    print(f"--- {c} ---"); display(top_values(c))

### 2.3 Trùng khóa

In [ ]:
dup = q(f'SELECT "{LOAN_ID}", COUNT(*) AS c FROM orig_s GROUP BY 1 HAVING c > 1 ORDER BY c DESC LIMIT 20')
print("Khoản vay xuất hiện > 1 lần:", len(dup)); display(dup)

### 2.4 Giá trị không hợp lệ (sửa quy tắc cho phù hợp)

In [ ]:
RULES_ORIG = {
    "fico_ngoai_300_850": "credit_score NOT BETWEEN 300 AND 850",
    "lai_suat_la":        "original_interest_rate <= 0 OR original_interest_rate > 20",
    "ltv_la":             "ltv < 1 OR ltv > 200",
    "dti_ngoai_1_65":     "dti < 1 OR dti > 65",
}
out = []
for name, cond in RULES_ORIG.items():
    try:
        n, tot = con.execute(f"SELECT COUNT(*) FILTER (WHERE {cond}), COUNT(*) FROM orig_s").fetchone()
        out.append((name, cond, n, round(100*n/tot, 3)))
    except Exception as e:
        out.append((name, cond, None, f"LỖI: {e}"))
display(pd.DataFrame(out, columns=["quy_tac", "dieu_kien", "so_dong_sai", "pct"]))

# first_payment_date có hợp lý so với quý khởi tạo không? (thường chỉ sau khởi tạo 1-3 tháng)
if has("first_payment_date"):
    display(q("""
    WITH t AS (SELECT quarter, first_payment_date,
                      CAST(SUBSTR(quarter,1,4) AS INT)*12 + CAST(SUBSTR(quarter,6,1) AS INT)*3 + 4 AS lim_idx
               FROM orig_s)
    SELECT quarter, MAX(first_payment_date) AS fp_max,
           COUNT(*) FILTER (WHERE (first_payment_date//100)*12 + first_payment_date%100 > lim_idx) AS tre_hon_4_thang
    FROM t GROUP BY 1 ORDER BY 1"""))

### 2.5 Ngoại lệ (IQR)

In [ ]:
res = []
for c in orig_num:
    n, tot, lo, hi = con.execute(f"""
        WITH s AS (SELECT quantile_cont("{c}",0.25) q1, quantile_cont("{c}",0.75) q3 FROM orig_s)
        SELECT COUNT(*) FILTER (WHERE "{c}" < q1-1.5*(q3-q1) OR "{c}" > q3+1.5*(q3-q1)), COUNT("{c}"),
               ANY_VALUE(q1-1.5*(q3-q1)), ANY_VALUE(q3+1.5*(q3-q1)) FROM orig_s, s""").fetchone()
    res.append((c, n, tot, round(100*n/tot, 2) if tot else None, lo, hi))
display(pd.DataFrame(res, columns=["cot", "so_ngoai_le", "so_non_null", "pct", "can_duoi", "can_tren"]).sort_values("pct", ascending=False))

## Bước 3. Phân tích đơn biến

### 3.1 Thống kê mô tả

In [ ]:
rows = []
for c in orig_num:
    rows.append(con.execute(f"""
        SELECT '{c}', COUNT("{c}"), AVG("{c}"), STDDEV("{c}"), MIN("{c}"), quantile_cont("{c}",0.25), MEDIAN("{c}"),
               quantile_cont("{c}",0.75), MAX("{c}"), skewness("{c}"), kurtosis("{c}") FROM orig_s""").fetchone())
stats_orig = pd.DataFrame(rows, columns=["cot","n","mean","std","min","q1","median","q3","max","skew","kurt"]).set_index("cot")
display(stats_orig)

### 3.2 Histogram (tính bằng SQL, chỉ kéo về vài chục dòng)

In [ ]:
def hist(col, bins=40, robust=True, ax=None):
    """robust=True: vẽ trong khoảng P1-P99 để ngoại lệ không làm méo biểu đồ."""
    if robust:
        e_lo, e_hi = f'quantile_cont("{col}",0.01)', f'quantile_cont("{col}",0.99)'
    else:
        e_lo, e_hi = f'MIN("{col}")', f'MAX("{col}")'
    lo, hi = con.execute(f"SELECT {e_lo}, {e_hi} FROM orig_s").fetchone()
    if lo is None: print(f"{col}: toàn NULL"); return
    if lo == hi: print(f"{col}: chỉ có 1 giá trị ({lo})"); return
    wd = (hi - lo) / bins
    df = q(f"""SELECT LEAST(FLOOR(("{col}"-{lo})/{wd}), {bins-1})::INT AS b, COUNT(*) AS n
               FROM orig_s WHERE "{col}" BETWEEN {lo} AND {hi} GROUP BY 1 ORDER BY 1""")
    ax = ax or plt.gca()
    ax.bar(lo + (df.b + 0.5) * wd, df.n, width=wd * 0.95)
    ax.set_title(col + (" (P1–P99)" if robust else "")); ax.set_ylabel("số dòng")

def hist_grid(cols, ncols=3, **kw):
    cols = list(cols); nrows = -(-len(cols) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.2*nrows)); axes = np.array(axes).ravel()
    for ax, c in zip(axes, cols): hist(c, ax=ax, **kw)
    for ax in axes[len(cols):]: ax.axis("off")
    plt.tight_layout(); plt.show()

hist_grid(orig_num)

### 3.3 Biến phân loại

In [ ]:
def bar_cat(col, k=15):
    df = q(f"""SELECT "{col}"::VARCHAR AS gia_tri, COUNT(*) AS n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 2) AS pct
               FROM orig_s GROUP BY 1 ORDER BY n DESC LIMIT {k}""")
    display(df)
    df.plot.barh(x="gia_tri", y="n", legend=False, title=col).invert_yaxis(); plt.show()

for c in orig_cat:
    nd = con.execute(f'SELECT COUNT(DISTINCT "{c}") FROM orig_s').fetchone()[0]
    if nd <= 200: bar_cat(c)
    else: print(f"{c}: {nd} giá trị khác nhau, bỏ qua (cardinality cao)")

## Bước 4. Phân tích hai biến và nhiều biến

### 4.1 Xu hướng theo quý

In [ ]:
trend_cols = [c for c in ["credit_score", "original_interest_rate", "ltv", "dti", "original_upb"] if c in orig_num]
agg = ", ".join(f'AVG("{c}") AS "{c}"' for c in trend_cols)
trend = q(f"SELECT quarter, COUNT(*) AS n, {agg} FROM orig_s GROUP BY 1 ORDER BY 1")
display(trend)
if len(trend) > 1 and trend_cols:
    trend.set_index("quarter")[trend_cols].plot(subplots=True, figsize=(10, 2.2*len(trend_cols)), marker="o")
    plt.tight_layout(); plt.show()
else:
    print("Chỉ có 1 quý: đặt ORIG_QUARTER = None để xem xu hướng.")

### 4.2 Ma trận tương quan

In [ ]:
corr_cols = orig_num[:12]
pairs = [(a, b) for i, a in enumerate(corr_cols) for b in corr_cols[i+1:]]
if pairs:
    sel = ", ".join(f'corr("{a}","{b}") AS "{a}||{b}"' for a, b in pairs)
    r = q(f"SELECT {sel} FROM orig_s").iloc[0]
    M = pd.DataFrame(np.eye(len(corr_cols)), index=corr_cols, columns=corr_cols)
    for k, v in r.items():
        a, b = k.split("||"); M.loc[a, b] = M.loc[b, a] = v
    fig, ax = plt.subplots(figsize=(7, 6)); im = ax.imshow(M, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(M))); ax.set_xticklabels(M.columns, rotation=60, ha="right")
    ax.set_yticks(range(len(M))); ax.set_yticklabels(M.index)
    for i in range(len(M)):
        for j in range(len(M)): ax.text(j, i, f"{M.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7)
    plt.colorbar(im); plt.title("Tương quan Pearson"); plt.tight_layout(); plt.show()

### 4.3 Scatter (lấy mẫu)

In [ ]:
x, y = ("credit_score", "original_interest_rate") if {"credit_score", "original_interest_rate"} <= set(orig_num) else orig_num[:2]
s = q(f'SELECT "{x}", "{y}" FROM orig_s USING SAMPLE 50000')
s.plot.scatter(x=x, y=y, alpha=0.1, s=3, title=f"{x} vs {y} (mẫu 50k dòng)"); plt.show()

### 4.4 So sánh theo nhóm (biến phân loại × biến số)

In [ ]:
for c in orig_cat:
    nd = con.execute(f'SELECT COUNT(DISTINCT "{c}") FROM orig_s').fetchone()[0]
    if nd > 20: continue
    agg = ", ".join(f'AVG("{v}") AS "{v}"' for v in trend_cols)
    print(f"--- theo {c} ---")
    display(q(f'SELECT "{c}" AS nhom, COUNT(*) AS n, {agg} FROM orig_s GROUP BY 1 ORDER BY n DESC'))

In [ ]:
# Cơ cấu mục đích vay theo quý (nếu có cột loan_purpose)
if has("loan_purpose"):
    d = q("""SELECT quarter, loan_purpose, COUNT(*) AS n FROM orig_s GROUP BY 1, 2""")
    p = d.pivot(index="quarter", columns="loan_purpose", values="n").fillna(0)
    (p.div(p.sum(axis=1), axis=0) * 100).plot.bar(stacked=True, figsize=(11, 4), title="Cơ cấu mục đích vay theo quý (%)")
    plt.ylabel("%"); plt.show()

## Bước 5. Tổng hợp insight (orig)

### Phát hiện chính
1. ...
2. ...
3. ...

### Hạn chế
- Mã đặc biệt (9999 / 999) đã đổi thành NULL: ...% dòng
- Cột thiếu nhiều: ...
- Giá trị bất thường: ...

### Việc cần làm tiếp
- [ ] Làm sạch và xuất `clean/orig`
- [ ] Chuyển sang `eda_perf.ipynb` để phân tích trả nợ và vỡ nợ

In [ ]:
# Xuất orig đã làm sạch (bỏ comment điều kiện WHERE khi đã chốt quy tắc). Mặc định KHÔNG chạy.
RUN_EXPORT = False
if RUN_EXPORT:
    con.execute("""
        COPY (SELECT * FROM orig_s WHERE credit_score BETWEEN 300 AND 850 AND original_interest_rate > 0)
        TO 'clean/orig' (FORMAT PARQUET, PARTITION_BY (quarter), COMPRESSION ZSTD, OVERWRITE_OR_IGNORE)
    """)
    print("Đã xuất clean/orig")